# 第 5 章：Q-learning 和 SARSA —— 第一个完整控制算法

> Ch04 学了**预测**（给定 $\pi$，估 $V^\pi$）。这一章学**控制**：找最优 $\pi^*$。
> 我们将用两个看起来几乎一样、本质却截然不同的算法，引出 RL 中最重要的概念之一：
> **on-policy vs off-policy**。

## 学习目标

1. 理解 **SARSA（on-policy）** 和 **Q-learning（off-policy）** 的更新规则
2. 推导两者的更新公式，并证明期望形式的等价性
3. 复现 Sutton-Barto **CliffWalk** 的经典对比图：SARSA 保守、Q-learning 激进
4. 理解 **maximization bias** 并掌握 **Double Q-learning**
5. 牢记 on/off-policy 的区别——它是 PPO / GRPO 设计的核心

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'rlenvs').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from utils import set_seed, plot_training_curve, plot_q_table, make_interactive
from rlenvs import CliffWalk, GridWorld

set_seed(0)

## 5.1 从预测到控制：为什么要用 $Q$ 而不是 $V$

Ch04 用 TD 学 $V$。但**控制**（找最优策略）需要比较**所有动作**的优劣，$V$ 不够用。

我们改用 **$Q(s, a)$**：在 $s$ 选 $a$、之后按 $\pi$ 行动的期望回报。一旦有了 $Q$，最优策略就是

$$
\pi^*(s) = \arg\max_a Q^*(s, a)
$$

### ε-greedy 探索

Ch01 的 ε-greedy 在这里复用：

$$
\pi(a|s) = \begin{cases} 1 - \epsilon + \epsilon/|\mathcal{A}| & a = \arg\max_{a'} Q(s, a') \\ \epsilon/|\mathcal{A}| & \text{otherwise} \end{cases}
$$

## 5.2 SARSA：on-policy TD 控制

### 算法

每一步采一个 transition $(S_t, A_t, R_{t+1}, S_{t+1}, A_{t+1})$（注意需要 $A_{t+1}$），然后更新：

$$
Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \big[ R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t) \big]
$$

名字来源：$S, A, R, S', A'$。

### 关键性质

**target 里的 $Q(S_{t+1}, A_{t+1})$ 中的 $A_{t+1}$ 是从 $\pi$ 采出来的**——也就是和 $A_t$ 同一个策略。

这就是 **on-policy**：用 $\pi$ 采的数据，去评估/改进 $\pi$ 本身。

## 5.3 Q-learning：off-policy TD 控制

### 算法

每步用 $(S_t, A_t, R_{t+1}, S_{t+1})$ 更新（**不需要 $A_{t+1}$**）：

$$
Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \big[ R_{t+1} + \gamma \max_{a'} Q(S_{t+1}, a') - Q(S_t, A_t) \big]
$$

**关键区别**：target 里是 $\max_{a'} Q(S_{t+1}, a')$，**不需要按 $\pi$ 采样**。

### off-policy 解释

虽然行为策略（用来采数据的）还是 ε-greedy，但 target 假设下一步会**走最大 Q 的动作**——也就是假设下一步是 **greedy 策略**。

所以 Q-learning 是 **off-policy**：行为策略 ≠ 目标策略。

### 期望形式：SARSA 和 Q-learning 的微妙差异

<details>
<summary><b>📝 期望形式等价：SARSA(λ=0) ≈ Expected SARSA ≈ Q-learning（点开看）</b></summary>

考虑 SARSA 在期望下（对 $A_{t+1}$ 求期望）：

$$
\mathbb{E}_{A_{t+1} \sim \pi}[Q(S_{t+1}, A_{t+1})] = \sum_a \pi(a | S_{t+1}) Q(S_{t+1}, a)
$$

这叫 **Expected SARSA**。它的方差比 SARSA 小（不用采样 $A_{t+1}$）。

如果 $\pi$ 是 greedy（即 $\pi(a|s) = \mathbb{1}[a = \arg\max Q]$），那么：

$$
\sum_a \pi(a|s) Q(s, a) = \max_a Q(s, a)
$$

**Expected SARSA 在 greedy 策略下 = Q-learning**。

所以 Q-learning ≈ SARSA 在"目标策略是 greedy"的特例。
</details>

## 5.4 实现：SARSA 和 Q-learning

In [ ]:
def epsilon_greedy_action(Q, s, epsilon):
    """ε-greedy 动作选择。"""
    if np.random.random() < epsilon:
        return np.random.randint(Q.shape[1])
    return int(np.argmax(Q[s]))


def sarsa(env, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    """SARSA 算法。"""
    nS, nA = env.nS, env.nA
    Q = np.zeros((nS, nA))
    episode_rewards = []
    for ep in range(n_episodes):
        s = env.reset()
        a = epsilon_greedy_action(Q, s, epsilon)
        done = False
        ep_reward = 0.0
        while not done:
            s_next, r, done, _ = env.step(a)
            ep_reward += r
            a_next = epsilon_greedy_action(Q, s_next, epsilon) if not done else 0
            td_target = r + (0 if done else gamma * Q[s_next, a_next])
            Q[s, a] += alpha * (td_target - Q[s, a])
            s, a = s_next, a_next
        episode_rewards.append(ep_reward)
    return Q, np.array(episode_rewards)


def q_learning(env, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    """Q-learning 算法。"""
    nS, nA = env.nS, env.nA
    Q = np.zeros((nS, nA))
    episode_rewards = []
    for ep in range(n_episodes):
        s = env.reset()
        done = False
        ep_reward = 0.0
        while not done:
            a = epsilon_greedy_action(Q, s, epsilon)
            s_next, r, done, _ = env.step(a)
            ep_reward += r
            td_target = r + (0 if done else gamma * np.max(Q[s_next]))
            Q[s, a] += alpha * (td_target - Q[s, a])
            s = s_next
        episode_rewards.append(ep_reward)
    return Q, np.array(episode_rewards)


# 验证两个算法在简单 GridWorld 上能学到东西
env = CliffWalk(seed=0)
Q_sarsa, rw_sarsa = sarsa(env, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1)
Q_ql, rw_ql = q_learning(env, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1)

print(f"SARSA 最后 50 episodes 平均奖励: {rw_sarsa[-50:].mean():.2f}")
print(f"Q-learning 最后 50 episodes 平均奖励: {rw_ql[-50:].mean():.2f}")

## 5.5 CliffWalk：SARSA 保守、Q-learning 激进

**这是 Sutton-Barto 最经典的图之一**。

4×12 CliffWalk：
- 起点 $(3, 0)$、终点 $(3, 11)$
- 中间 row=3, col=1..10 是悬崖，落入 -100，回到起点
- 每步 -1（鼓励尽快到达）

**直觉预期**：
- **Q-learning 学到 "贴着悬崖边走最短路"**——因为 target 假设下一步 greedy（不会掉下去），但行为 ε-greedy 偶尔会掉下去
- **SARSA 学到 "远离悬崖的更安全路径"**——因为它考虑 ε-greedy 下可能掉下去的事实

In [ ]:
# 跑 30 个 seed，对比
n_seeds = 30
n_eps = 500
all_rw_sarsa = np.zeros((n_seeds, n_eps))
all_rw_ql = np.zeros((n_seeds, n_eps))
for seed in range(n_seeds):
    env_s = CliffWalk(seed=seed)
    _, rw_s = sarsa(env_s, n_episodes=n_eps, alpha=0.5, epsilon=0.1)
    env_q = CliffWalk(seed=seed)
    _, rw_q = q_learning(env_q, n_episodes=n_eps, alpha=0.5, epsilon=0.1)
    all_rw_sarsa[seed] = rw_s
    all_rw_ql[seed] = rw_q

# 画两条曲线
fig, ax = plt.subplots(figsize=(9, 5))
sm_sarsa = np.convolve(all_rw_sarsa.mean(0), np.ones(20)/20, mode='valid')
sm_ql = np.convolve(all_rw_ql.mean(0), np.ones(20)/20, mode='valid')
ax.plot(sm_sarsa, label='SARSA (on-policy)', linewidth=2)
ax.plot(sm_ql, label='Q-learning (off-policy)', linewidth=2)
ax.set_xlabel('episode (smoothed w=20)')
ax.set_ylabel('reward')
ax.set_title('CliffWalk：SARSA 更稳，Q-learning 更激进')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"SARSA 渐进奖励: {all_rw_sarsa[:, -50:].mean():.2f}")
print(f"Q-learning 渐进奖励: {all_rw_ql[:, -50:].mean():.2f}")
print(f"最优值（贴悬崖走）: -12")

你应该看到：

- **训练期**：Q-learning 经常掉悬崖（reward -100 频繁），SARSA 更稳
- **渐进 reward**：Q-learning ≈ -12（最优路径）但方差大；SARSA ≈ -25~−30（绕远更稳）

**核心洞察**：
- Q-learning 学的 $\pi^*$ 是"如果 ε=0 的最优"
- SARSA 学的 $\pi^*$ 是"考虑 ε 探索成本的最优"
- ε 越小，两者越接近

## 5.6 看一眼 SARSA 学到的策略

我们把学到的 $\pi$ 画出来，看 SARSA 是不是真的"绕开了悬崖"。

In [ ]:
def render_cliff_policy(Q, ax, title='Learned policy'):
    """在 CliffWalk 上画策略。"""
    ax.clear()
    n_rows, n_cols = 4, 12
    # 悬崖
    from matplotlib.patches import Rectangle, Circle
    for c in range(1, 11):
        ax.add_patch(Rectangle((c, 3), 1, 1, color='crimson', alpha=0.4))
    # 终点
    ax.add_patch(Rectangle((11, 3), 1, 1, color='gold', alpha=0.7))
    # 起点
    # 画策略箭头
    arrows = {0: (0, 0.4), 1: (0.4, 0), 2: (0, -0.4), 3: (-0.4, 0)}
    for s in range(48):
        r, c = divmod(s, 12)
        if r == 3 and 1 <= c <= 10:  # 悬崖
            continue
        if s == 47:  # 终点
            continue
        a = int(np.argmax(Q[s]))
        dr, dc = arrows[a]
        ax.arrow(c + 0.5, r + 0.5, dc, -dr, head_width=0.15, head_length=0.1, fc='navy', ec='navy')
    ax.set_xlim(0, 12); ax.set_ylim(0, 4); ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title)

env = CliffWalk(seed=0)
Q_sarsa, _ = sarsa(env, n_episodes=2000, alpha=0.5, epsilon=0.1)
env = CliffWalk(seed=0)
Q_ql, _ = q_learning(env, n_episodes=2000, alpha=0.5, epsilon=0.1)

fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
render_cliff_policy(Q_sarsa, axes[0], title='SARSA 学到的策略（绕远路）')
render_cliff_policy(Q_ql, axes[1], title='Q-learning 学到的策略（贴悬崖）')
plt.tight_layout(); plt.show()

## 5.7 Maximization Bias：为什么 Q-learning 会"过度乐观"

Q-learning 的 target 里有一个 $\max$。**max 操作会引入正偏差**。

### 直觉

假设真值 $Q^*(s, a) = 0$ 对所有 $a$，但你的估计 $Q(s, a)$ 有噪声（围绕 0 抖）。

$\max_a Q(s, a)$ 的期望 = $\mathbb{E}[\max_a Q(s, a)] > \max_a \mathbb{E}[Q(s, a)] = 0$

也就是"max 的期望 > 期望的 max"。这叫 **Jensen 不等式**。

后果：Q-learning 系统性高估某些 $(s, a)$，可能反复去试那些被高估的次优动作。

### 一个最小例子

考虑一个状态 $s$ 有两个动作 $a_1, a_2$：
- $a_1$：奖励 $\sim N(0, 1)$（其实 0 期望）
- $a_2$：奖励 $\sim N(-0.1, 1)$（其实稍微负）

**真实最优**是 $a_1$（期望 0）。但 Q-learning 可能在某次采样里看到 $a_2$ 偶然给个大正数，把它的 $Q$ 高估，然后反复去试 $a_2$。

> 🤔 **先猜再跑**：下面这个实验会跑 200 个 seed、每个 300 episode，统计 Q-learning **最后 50 个 episode 里选 $a_2$（次优动作）的频率**。你猜是接近 0%（学明白了）、5%（偶尔犯迷糊）、还是 30%+（系统性偏差）？
>
> <details><summary>写下你的百分比再点开</summary>
>
> 关键在于 max：两个动作的 Q 都有噪声，而 `max` 永远挑**当下看起来更高**的那个——噪声里"虚高"的一侧更容易被选中。这不是偶尔犯迷糊，是结构性偏袒。猜 5% 的读者，准备被结果惊讶。
> </details>

In [ ]:
class OneStateTrap:
    """单状态、两动作的 trap。"""
    def __init__(self):
        self.nS = 1
        self.nA = 2
        self._rng = np.random.default_rng()
    def reset(self):
        return 0
    def step(self, a):
        # a=0: N(0,1)；a=1: N(-0.1, 1)
        r = self._rng.normal(0, 1) if a == 0 else self._rng.normal(-0.1, 1)
        return 0, r, True, {}


def q_learning_trap(n_episodes=300, alpha=0.1, gamma=0.0, epsilon=0.1):
    env = OneStateTrap()
    Q = np.zeros((1, 2))
    a2_frequencies = []
    for ep in range(n_episodes):
        s = env.reset()
        done = False
        actions_taken = []
        while not done:
            a = epsilon_greedy_action(Q, s, epsilon)
            actions_taken.append(a)
            s_next, r, done, _ = env.step(a)
            td_target = r  # γ=0, 没有下一状态
            Q[s, a] += alpha * (td_target - Q[s, a])
            s = s_next
        a2_frequencies.append(np.mean(actions_taken))
    return Q, np.array(a2_frequencies)


# 跑 1000 个 seed
n_seeds = 500
freq_a2 = np.zeros((n_seeds, 300))
for seed in range(n_seeds):
    np.random.seed(seed)
    Q, freq = q_learning_trap(n_episodes=300, alpha=0.1, epsilon=0.1)
    freq_a2[seed] = freq

print(f"最优 a=0 的频率应该是 0.5（ε=0.1 + 90% greedy 选 a=0）")
print(f"实际 Q-learning 选 a=1 的平均频率（最后 50 episodes）：{freq_a2[:, -50:].mean():.3f}")
print(f"Q[0, 0] 期望 ~ 0, Q[0, 1] 期望 ~ -0.1")
print(f"实际 Q[0]: {Q}")

# 画频率曲线
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.convolve(freq_a2.mean(0), np.ones(20)/20, mode='valid'),
        label='Q-learning: 频率 a=1', linewidth=2)
ax.axhline(0.1 * 0.5, color='gray', linestyle='--', label='理论下界 ε * 0.5')
ax.set_xlabel('episode (smoothed)')
ax.set_ylabel('P(a=1)')
ax.set_title('Maximization Bias：Q-learning 高估了 a=1')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5.8 Double Q-learning：解决方案

**思路**：把经验随机分两半，维护两个独立的 $Q_1, Q_2$。每步用其中一个评估、用另一个选择动作：

```
以 0.5 概率更新 Q_1：
    a* = argmax_a Q_1[s', a]
    Q_1[s, a] += α [r + γ Q_2[s', a*] - Q_1[s, a]]
否则更新 Q_2：
    a* = argmax_a Q_2[s', a]
    Q_2[s, a] += α [r + γ Q_1[s', a*] - Q_2[s, a]]
```

**关键**：用 $Q_1$ 选动作、用 $Q_2$ 评估，两者独立→期望上消除了 max 的偏差。

## 5.9 📝 练习：实现 Double Q-learning

**任务**：

1. 实现 `double_q_learning(env, ...)` 函数
2. 在上面的 OneStateTrap 上验证它消除了 maximization bias（选 a=1 的频率应该接近 ε/2 = 0.05）
3. 在 CliffWalk 上对比 Q-learning 和 Double Q-learning

**预期结果**：
- Double Q-learning 在 trap 上选 a=1 的频率从 ~25% 降到 ~5%
- 在 CliffWalk 上 Double Q-learning 性能略好（方差更小）

> 参考答案：`solutions/ch05_double_q_learning.ipynb`

## 5.10 小结

| | SARSA | Q-learning |
|---|---|---|
| 策略 | on-policy | off-policy |
| Target | $R + \gamma Q(s', a')$ | $R + \gamma \max_{a'} Q(s', a')$ |
| 行为策略 | $\epsilon$-greedy（与目标同） | $\epsilon$-greedy（与目标不同） |
| CliffWalk | 保守绕远 | 激进贴崖 |
| 偏差 | 较小 | 有 maximization bias |

### on-policy vs off-policy（**重要！**）

> **on-policy**：用 $\pi$ 采的数据，训练 $\pi$ 自己
> **off-policy**：用一个 behavior 策略采的数据，训练另一个 target 策略

这个区别会在后面的 PPO / GRPO 中反复出现：

- **PPO 是 on-policy**：必须用当前 $\pi_\theta$ 采的数据更新 $\pi_\theta$。每次更新后数据"作废"。
- **Q-learning / DQN 是 off-policy**：可以用任意策略采的数据训练目标策略。**经验回放（replay buffer）成为可能**——大幅提高样本效率。

**为什么 PPO 在 LLM 上更好？** Ch13 我们会看到，GRPO 之所以取代 PPO 的"LLM 版本"，正是因为它进一步简化了 on-policy 训练的复杂度。

---

## Phase 1 结束！

你已经学完了经典 RL 的核心：

- ✅ Ch00 RL 全景
- ✅ Ch01 多臂老虎机：探索 vs 利用
- ✅ Ch02 MDP + 贝尔曼方程
- ✅ Ch03 动态规划：精确求解
- ✅ Ch04 TD 学习：从样本中学习
- ✅ Ch05 SARSA / Q-learning：on-policy vs off-policy

接下来 **Phase 2**：

- **Ch05b PyTorch 速成（没用过 PyTorch 的读者先读这个，1 小时）**
- Ch06 DQN + 函数逼近（神经网络 + 经验回放）
- Ch07 策略梯度定理
- Ch08 Actor-Critic + GAE
- Ch09 TRPO + PPO（**整个 Phase 2 的核心**）

然后 **Phase 3**：

- Ch13 GRPO（DeepSeek-R1 的核心算法）

请你**在跳到 Phase 2 之前**，确保：

1. 能默写出贝尔曼期望方程（Ch02）
2. 能用一两句话解释 on-policy vs off-policy（Ch05）
3. 知道 TD(0) 和 MC 的核心区别（Ch04）

这三个知识点是后面所有内容的钥匙。更完整的门槛自测见 `STUDY_GUIDE.md` 的"Phase 1 → 2 门槛"。